# 01. NetHealth 전처리

원본 CSV 두 개(활동, 수면 조각)가 **하루 한 줄 표**가 되기까지를 규칙 순서대로 확인함.
규칙은 2026-10-13 회의에서 확정했고, README의 '전처리 규칙' 표와 같음.

| 순서 | 규칙 | 함수 |
| --- | --- | --- |
| 1 | 중복 `(egoid, datadate)` → 빈칸 가장 적은 줄 하나 | `pp.drop_duplicate_days` |
| 2 | 착용률 80% 미만인 날의 활동·심박 → 빈칸(NaN). 수면은 유지 | `pp.build_activity` |
| 3~5 | 총 수면 = 조각 합산 / 잠든 시각 = 90분 안 조각 묶은 뒤 가장 긴 덩어리의 시작 (정오부터 분) | `pp.merge_sleep` |
| 7 | 기록 있는 날 42일 미만인 사람 제외 | `pp.filter_min_days` |
| 8 | 빈칸 채우기(보간, ffill, bfill) 금지 | (어디에서도 안 함) |
| 11 | 연구 기간 나누기: 1~2학년 / 3~4학년 (경계 2017-08-13) | `pp.split_study_phase` |
| - | 학사 달력: study_week, term_tag(학기/방학), event_tag(개강주/시험/휴일/없음) | `dl.load_calendar` |

데이터는 `data/nethealth/` 아래에 있어야 함 (깃허브에는 올리지 않음).

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 한글 글꼴 (윈도우: 맑은 고딕, 맥: AppleGothic, 그 외: 나눔고딕)
for f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if f in {x.name for x in font_manager.fontManager.ttflist}:
        plt.rcParams['font.family'] = f
        break
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.width', 160)

from src import data_loader as dl, preprocessing as pp, baseline as bl, evaluation as ev

## 0. 원본 읽기

In [ ]:
A, S = dl.read_raw()
print('활동 원본:', A.shape, '| 사람 수:', A['egoid'].nunique(), '| 기간:', A['datadate'].min(), '~', A['datadate'].max())
print('수면 원본(조각):', S.shape, '| 사람 수:', S['egoid'].nunique())
A.head(3)

## 1. 중복 제거
같은 사람·같은 날이 여러 줄이면 빈칸이 가장 적은 줄 하나만 남김. 빈칸 수가 같으면 파일에서 먼저 나온 줄.

In [ ]:
dup = A.duplicated(['egoid', 'datadate'], keep=False)
print('중복에 걸린 줄:', int(dup.sum()), '| 중복된 (사람, 날) 수:', A[dup].groupby(['egoid', 'datadate']).ngroups)
A1 = pp.drop_duplicate_days(A)
print('중복 제거 후:', len(A), '→', len(A1))

## 2. 착용률 80% 미만 → 활동·심박 빈칸
줄을 지우지 않고 값만 비움. 그래야 그날의 수면 기록은 그대로 쓸 수 있음. 0으로 채우면 '하나도 안 걸었다'가 되어 버리니 반드시 빈칸.

In [ ]:
act = pp.build_activity(A)
low = act['complypercent'] < pp.WEAR_MIN
print(f'착용 80% 미만인 날: {low.sum():,}일 ({low.mean():.1%})')
print('그 날들의 steps가 모두 빈칸인지:', act.loc[low, 'steps'].isna().all())

fig, ax = plt.subplots(figsize=(7, 3))
act['complypercent'].plot.hist(bins=50, ax=ax)
ax.axvline(pp.WEAR_MIN, color='r', ls='--', label='80% 기준')
ax.set_xlabel('착용률(%)'); ax.set_ylabel('날 수'); ax.legend(); plt.show()

## 3~5. 수면 조각 합치기
- `sleep_min` = 그날 조각 **모두** 더한 총 수면 (낮잠 포함)
- `bed_min` = 90분 안에 이어지는 조각을 한 덩어리로 묶은 뒤, **가장 오래 잔 덩어리의 시작 시각**
- 단위 = **정오부터 몇 분** (23:30 → 690, 01:00 → 780). 자정을 넘어도 숫자가 이어져서 평균·z 계산이 쉬움.

In [ ]:
sl = pp.merge_sleep(S)
print('하루 수면 표:', sl.shape)
print('하루 조각 수:', sl['n_episodes'].value_counts().sort_index().head(6).to_dict())

# 조각이 여러 개인 날 예시
ex = sl[sl['n_episodes'] >= 3].iloc[0]
raw = S[(S['egoid'].astype(str) == ex['pid']) & (pd.to_datetime(S['dataDate']) == ex['date'])]
display(raw[['egoid', 'dataDate', 'timetobed', 'timeoutofbed', 'bedtimedur', 'minsasleep']])
ex

In [ ]:
def clock(m):
    """정오부터 분 → 'HH:MM'"""
    m = (m + 720) % 1440
    return f'{int(m // 60):02d}:{int(m % 60):02d}'

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
sl['sleep_min'].clip(0, 900).plot.hist(bins=60, ax=axes[0]); axes[0].set_xlabel('총 수면(분)')
sl['bed_min'].plot.hist(bins=96, ax=axes[1]); axes[1].set_xlabel('잠든 시각')
ticks = range(0, 1441, 180)
axes[1].set_xticks(list(ticks), [clock(t) for t in ticks])
plt.tight_layout(); plt.show()
print('잠든 시각 중앙값:', clock(sl['bed_min'].median()))

## 7. 기록 42일 미만인 사람 제외
기준선 28일 + 평가 14일 = 42일. '기록 있는 날' = 걸음이나 수면 중 하나라도 있는 날.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    all_people = dl.load_nethealth(min_days=None)
nh = dl.load_nethealth()      # 기본값: 위 규칙 전부 적용. Calendar.csv가 있으면 달력 열도 붙음
print('42일 필터 전:', all_people['pid'].nunique(), '명 →  후:', nh['pid'].nunique(), '명')
has = nh[['steps', 'sleep_min']].notna().any(axis=1).groupby(nh['pid']).sum()
print('사람별 기록 일수: 중앙값', int(has.median()), '| 최소', int(has.min()), '| 최대', int(has.max()))

## 완성된 하루 표

In [ ]:
print(nh.shape)
nh.head()

In [ ]:
summary = pd.DataFrame({'빈칸 비율': nh.isna().mean().round(3)}).join(nh.describe().T[['mean', '50%', 'min', 'max']].round(1))
summary

빈칸은 **채우지 않음** (보간·ffill·bfill 모두 금지). 미래 값이 과거 계산에 섞이기 때문임. 한 사람의 걸음을 그려 보면 빈 날은 선이 끊겨 있음.

In [ ]:
pid = has.sort_values().index[len(has) // 2]          # 기록 일수가 중간쯤인 사람
one = nh[nh['pid'] == pid].set_index('date').asfreq('D')
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
one['steps'].plot(ax=axes[0], marker='.', lw=0.8); axes[0].set_ylabel('steps')
one['bed_min'].plot(ax=axes[1], marker='.', lw=0.8, color='C1'); axes[1].set_ylabel('bed_min')
axes[0].set_title(f'pid {pid} — 빈 날은 비워 둠'); plt.tight_layout(); plt.show()

### 요일 (`dow`, 0=월 ~ 6=일)
요일마다 평소 값이 다름. 02 노트북에서 `personal_z(..., dow_adjust=True)`로 요일 보정을 켤 수 있음.

In [ ]:
nh.groupby('dow')[['steps', 'sleep_min', 'bed_min']].median().rename(index=dict(enumerate('월화수목금토일'))).round(0)

## 11. 연구 기간 나누기 (민감도 분석용)
- `period='before'` = 1~2학년, `period='after'` = 3~4학년
- 경계 `2017-08-13` = Calendar 104주차(3학년 오리엔테이션 주) 시작. 2학년 이후 참여 인원이 크게 줄어(NetHealth Table 1 Attrition Counts) 데이터 성격이 바뀌는 시점임
- 기기 교체 날짜는 공식 문서에서 확인되지 않아 경계로 쓰지 않음

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    before = dl.load_nethealth(period='before')
    after = dl.load_nethealth(period='after')
print(f"경계 {pp.PHASE_CUTOFF}: 1~2학년 {before['pid'].nunique()}명 {len(before):,}일 / 3~4학년 {after['pid'].nunique()}명 {len(after):,}일")

## 학사 달력 (study_week, term_tag, event_tag)
`data/nethealth/Calendar.csv`가 있으면 `load_nethealth`가 세 열을 자동으로 붙임. 없으면 경고만 내고 진행함.
- `term_tag`: `학기` = 개강일 ~ 그 학기 기말고사 끝난 날, 나머지는 `방학`
- `event_tag`: `개강주`(개강일부터 7일) / `시험` / `휴일`(break, Thanksgiving, Easter, Christmas) / `없음`. 겹치면 시험 > 휴일 > 개강주
- 달력 범위(2015-08-16 ~ 2019-05-11) 밖의 날은 빈칸

In [ ]:
if 'term_tag' in nh.columns:
    display(nh['term_tag'].value_counts(dropna=False).rename('날 수').to_frame())
    display(nh['event_tag'].value_counts(dropna=False).rename('날 수').to_frame())
    display(nh.groupby('event_tag')[['steps', 'sleep_min', 'bed_min']].median().round(0))
else:
    print('Calendar.csv가 없어 달력 열이 없습니다')